# this is for intrasubject results

In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import pandas as pd
import numpy as np

# load data


In [ ]:
# other methods
other_df = pd.read_csv('examples/results/other_results_plus2.csv')
model_name_map = {
    'SpaGCN_without': 'SpaGCN',
    'conST_nopre': 'conST',
    'louvain': 'Louvain',
    'leiden': 'Leiden',
    'DeepST': 'DeepST',
    'STAGATE': 'STAGATE',
    'SpaceFlow': 'SpaceFlow',
    'CCST': 'CCST',
    'SEDR': 'SEDR',
    'BASS': 'BASS',
    'SpaGCN_with': 'SpaGCN(HE)',
    'SCAN-IT': 'SCAN-IT',
    'DeepST': 'GraphST',
    'stLearn': 'stLearn',
    'BayesSpace': 'BayesSpace',
    'IRIS': 'IRIS',
    'Seurat': 'Seurat'
}
other_df['Algorithm'] = other_df.loc[:, 'Method'].map(model_name_map)

zeroshot_df = pd.read_csv('examples/results/zeroshot_all_metrics_all_replicates.csv')
finetune_df = pd.read_csv('examples/results/finetunePro_all_metrics_all_replicates.csv')
map_data_type = {
    'visium': 'Visium',
    'starmap': 'STARmap',
    'merfish': 'MERFISH'
}
zeroshot_df['data_type'] = zeroshot_df.loc[:, 'data_type'].map(map_data_type)
finetune_df['data_type'] = finetune_df.loc[:, 'data_type'].map(map_data_type)






In [ ]:
len(other_df.Method.unique())

In [ ]:
finetune_df

# table for comparing zeroshot and finetune and other methods

In [ ]:
# remove unneeded data for this table
# this is for intrasubject results
# remove my data
remove_data_names = ["151507", "151508", "151509", "151510", "151669","151670", "151671", "151672"]
finetune_df_sub = finetune_df[~finetune_df.data_name.isin(remove_data_names)].copy()



# remove other data

select_type = ['Visium', 'STARmap', 'MERFISH']
other_df_sub = other_df[other_df.Biotech.isin(select_type)].copy()
remove_data_names = ["151507", "151508", "151509", "151510", "151669","151670", "151671", "151672"]
remove_data_names.append('151673')  # because it is used for finetune
other_df_sub = other_df_sub[~other_df_sub.DataName.isin(remove_data_names)]

zeroshot_df_sub = zeroshot_df[~zeroshot_df.data_name.isin(remove_data_names)].copy()


In [ ]:
matrice_keys = ['NMI', 'HOM', 'COM', 'CHAOS', 'PAS', 'ASW']
for matrice_key in matrice_keys:
    my_mean = (finetune_df_sub.groupby(['data_type', 'result_type','stage']).agg({matrice_key: 'mean'})
    .unstack(level=0)
    .droplevel(0, axis=1)
    )
    zeroshot_mean = (zeroshot_df_sub.groupby(['data_type', 'model_name']).agg({matrice_key: 'mean'})
    .unstack(level=0)
    .droplevel(0, axis=1)
    )
    zeroshot_mean.index = ['LLMiniST-Z', 'GPT-4o', 'GPT-4o-mini']
    index_names = ['LLMiniST-F', 'LLMiniST-Fs', 'LLMiniST-F (val)']
    my_mean.index = index_names
    my_mean = pd.concat([my_mean, zeroshot_mean])

    other_mean= (other_df_sub.groupby(['Biotech', 'Algorithm'])
    .agg({matrice_key: 'mean'})
    .unstack(level=0)
    .droplevel(0, axis=1)
    )

    my_mean = my_mean.loc[['LLMiniST-F', 'LLMiniST-Fs','LLMiniST-Z', 'LLMiniST-F (val)'], :]
    all_mean = pd.concat([other_mean, my_mean])
    # Calculate ranks for each column, ascending=False because higher NMI is better

    if matrice_key == 'CHAOS' or matrice_key == 'PAS':
        ranks = all_mean.apply(lambda x: x.rank(ascending=True, method='average', na_option='keep'))
    else:
        ranks = all_mean.apply(lambda x: x.rank(ascending=False, method='average', na_option='keep'))

    # Calculate mean rank across columns, ignoring NaN values
    avg_ranks = ranks.mean(axis=1, skipna=True)

    # Add the average a new column
    all_mean['Avg.'] = all_mean.mean(axis=1)

    # Add the average rank as a new column
    all_mean['Rank'] = avg_ranks

    # Sort by average rank
    all_mean_sorted = all_mean.sort_values('Rank')
    # Round all numeric columns to 4 decimal places
    all_mean_sorted = all_mean_sorted.round(4)

    my_std = (finetune_df_sub.groupby(['data_type', 'result_type','stage']).agg({matrice_key: 'std'})
    .unstack(level=0)
    .droplevel(0, axis=1)
    ).round(4)
    my_std.index = index_names
    zeroshot_std = (zeroshot_df_sub.groupby(['data_type', 'model_name']).agg({matrice_key: 'std'})
    .unstack(level=0)
    .droplevel(0, axis=1)
    )
    zeroshot_std.index = ['LLMiniST-Z', 'GPT-4o', 'GPT-4o-mini']
    my_std = pd.concat([my_std, zeroshot_std])
    my_std = my_std.loc[['LLMiniST-F', 'LLMiniST-Fs','LLMiniST-Z', 'LLMiniST-F (val)'], :]
    other_std = (other_df_sub.groupby(['Biotech', 'Algorithm'])
    .agg({matrice_key: 'std'})
    .unstack(level=0)
    .droplevel(0, axis=1)
    ).round(4)

    all_std = pd.concat([other_std, my_std])
    all_std_sorted =all_std.loc[all_mean_sorted.index, :]

    # Create a formatted table combining means and standard deviations
    def format_mean_std(mean, std):
        return f"{mean:.3f}±{std:.3f}"

    # Create combined table
    formatted_table = pd.DataFrame(index=all_mean_sorted.index)

    # Format each dataset column with mean ± std
    for col in [ 'STARmap', 'Visium', 'MERFISH']:
        formatted_table[col] = [
            format_mean_std(mean, std) 
            for mean, std in zip(all_mean_sorted[col], all_std_sorted[col])
        ]

    # Add average and rank columns
    formatted_table['Avg.'] = all_mean_sorted['Avg.'].round(3).astype(str)
    formatted_table['Rank'] = (all_mean_sorted['Rank']-1).round(1).astype(str) # -1 because the val is included

    # Display the formatted table
    print(formatted_table.to_string())

    # Optionally save to LaTeX format for paper
    formatted_table.to_latex(f'tables/raw_{matrice_key}_alltypedata_comparison_intrasubject_table.tex', escape=False)

    

In [ ]:
my_mean = (finetune_df_sub.groupby(['data_type', 'result_type','stage']).agg({matrice_key: 'mean'})
 .unstack(level=0)
 .droplevel(0, axis=1)
)


In [ ]:
zeroshot_mean = (zeroshot_df_sub.groupby(['data_type', 'model_name']).agg({matrice_key: 'mean'})
 .unstack(level=0)
 .droplevel(0, axis=1)
)
zeroshot_mean.index = ['Gemini 1.5 Pro', 'GPT-4o', 'GPT-4o-mini']

In [ ]:
# rename index of my_df
index_names = ['LLMiniST-F', 'LLMiniST-Fs', 'LLMiniST-F (val)']
my_mean.index = index_names
my_mean = pd.concat([my_mean, zeroshot_mean])
my_mean


In [ ]:
other_mean= (other_df_sub.groupby(['Biotech', 'Algorithm'])
 .agg({matrice_key: 'mean'})
 .unstack(level=0)
 .droplevel(0, axis=1)
)

my_mean = my_mean.loc[['LLMiniST-F', 'LLMiniST-Fs','Gemini 1.5 Pro'], :]



In [ ]:
all_mean = pd.concat([other_mean, my_mean])
# Calculate ranks for each column, ascending=False because higher NMI is better
ranks = all_mean.apply(lambda x: x.rank(ascending=False, method='average', na_option='keep'))

# Calculate mean rank across columns, ignoring NaN values
avg_ranks = ranks.mean(axis=1, skipna=True)

# Add the average a new column
all_mean['Avg.'] = all_mean.mean(axis=1)

# Add the average rank as a new column
all_mean['Rank'] = avg_ranks

# Sort by average rank
all_mean_sorted = all_mean.sort_values('Rank')
# Round all numeric columns to 4 decimal places
all_mean_sorted = all_mean_sorted.round(4)

print(all_mean_sorted)

In [ ]:
my_std = (finetune_df_sub.groupby(['data_type', 'result_type','stage']).agg({matrice_key: 'std'})
 .unstack(level=0)
 .droplevel(0, axis=1)
).round(4)
my_std.index = index_names
zeroshot_std = (zeroshot_df_sub.groupby(['data_type', 'model_name']).agg({matrice_key: 'std'})
 .unstack(level=0)
 .droplevel(0, axis=1)
)
zeroshot_std.index = ['Gemini 1.5 Pro', 'GPT-4o', 'GPT-4o-mini']
my_std = pd.concat([my_std, zeroshot_std])
my_std = my_std.loc[['LLMiniST-F', 'LLMiniST-Fs','Gemini 1.5 Pro'], :]
other_std = (other_df_sub.groupby(['Biotech', 'Algorithm'])
 .agg({matrice_key: 'std'})
 .unstack(level=0)
 .droplevel(0, axis=1)
).round(4)

all_std = pd.concat([other_std, my_std])
all_std_sorted =all_std.loc[all_mean_sorted.index, :]


In [ ]:
# Create a formatted table combining means and standard deviations
def format_mean_std(mean, std):
    return f"{mean:.3f}±{std:.3f}"

# Create combined table
formatted_table = pd.DataFrame(index=all_mean_sorted.index)

# Format each dataset column with mean ± std
for col in ['MERFISH', 'STARmap', 'Visium']:
    formatted_table[col] = [
        format_mean_std(mean, std) 
        for mean, std in zip(all_mean_sorted[col], all_std_sorted[col])
    ]

# Add average and rank columns
formatted_table['Avg.'] = all_mean_sorted['Avg.'].round(3).astype(str)
formatted_table['Rank'] = all_mean_sorted['Rank'].round(1).astype(str)

# Display the formatted table
print(formatted_table.to_string())

# Optionally save to LaTeX format for paper
formatted_table.to_latex('tables/raw_{matrice_key}_alltypedata_comparison_table.tex', escape=False)